In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/rag-from-scratch/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 05 · Evaluation

You cannot improve what you don't measure, and "the demo looked good" is not
measurement. With the labelled `qrels` we can score each retriever with two
standard metrics:

* **Recall@k** — is a gold document in the top *k*? (Did we even fetch it?)
* **MRR** — 1/rank of the first gold hit. (How high did it land?)


In [ ]:
# --- setup: make `import ragkit` work from notebooks/ or solutions/ ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from ragkit.corpus import load_documents, load_corpus, load_qrels, tokenize
from ragkit.embed import get_embedder
from ragkit import llm

### Exercise 1 — the metrics

Implement both over **document-level** rankings (gold labels are per document).
These are pure functions — the asserts pin the exact maths.


In [ ]:
def recall_at_k(ranked_docs, gold_docs, k):
    top = ranked_docs[:k]
    return 1.0 if any(g in top for g in gold_docs) else 0.0

def mrr(ranked_docs, gold_docs):
    for rank, d in enumerate(ranked_docs, start=1):
        if d in gold_docs:
            return 1.0 / rank
    return 0.0

assert recall_at_k(["a", "b", "c"], ["c"], k=3) == 1.0
assert recall_at_k(["a", "b", "c"], ["c"], k=2) == 0.0
assert mrr(["a", "b", "c"], ["b"]) == 0.5
assert mrr(["a", "b", "c"], ["z"]) == 0.0
print("metrics OK")

### Exercise 2 — the evaluation harness

`evaluate` takes a retrieval function `run(query) -> ranked list of doc_ids` and
averages Recall@k and MRR over a set of questions. Fill the averaging loop; the
assert uses a fake `run` with a known answer so it's model-independent.


In [ ]:
def evaluate(run, questions, k=3):
    rec = mr = 0.0
    for q in questions:
        ranked = run(q["question"])
        rec += recall_at_k(ranked, q["gold_docs"], k)
        mr += mrr(ranked, q["gold_docs"])
    n = len(questions)
    return {"recall@%d" % k: rec / n, "mrr": mr / n}

# fake retriever: always returns the same ranking, so the score is hand-checkable
fake_qs = [{"question": "x", "gold_docs": ["b"]},
           {"question": "y", "gold_docs": ["z"]}]   # one hit at rank 2, one miss
res = evaluate(lambda q: ["a", "b", "c"], fake_qs, k=3)
assert abs(res["recall@3"] - 0.5) < 1e-9    # 1 of 2 has a gold doc in top-3
assert abs(res["mrr"] - 0.25) < 1e-9        # (1/2 + 0) / 2
print("harness OK:", res)

### The payoff — compare the retrievers you built

Wire up dense, BM25, and hybrid (RRF) over the section chunks, then score them
overall and broken down by question kind. This is where the earlier notebooks
prove themselves.


In [ ]:
from ragkit.reference import (structure_aware_chunks, chunk_corpus, BM25,
                              reciprocal_rank_fusion, to_doc_ranking)
docs = load_documents()
chunks = chunk_corpus(docs, structure_aware_chunks)
emb = get_embedder()
mat = emb.encode([c.text for c in chunks])
bm = BM25([tokenize(c.text) for c in chunks])
ids = [c.chunk_id for c in chunks]

def dense_run(q):
    order = np.argsort(-(mat @ emb.encode(q)))
    return to_doc_ranking([ids[i] for i in order])

def bm25_run(q):
    ranked = bm.search(q, k=len(chunks))
    return to_doc_ranking([ids[i] for i, _ in ranked])

def hybrid_run(q):
    d = [ids[i] for i in np.argsort(-(mat @ emb.encode(q)))[:10]]
    b = [ids[i] for i, _ in bm.search(q, 10)]
    fused = [cid for cid, _ in reciprocal_rank_fusion([d, b])]
    return to_doc_ranking(fused)

qrels = load_qrels()
runs = {"dense": dense_run, "bm25": bm25_run, "hybrid": hybrid_run}
print(f"{'method':8} {'recall@3':>9} {'mrr':>6}")
for name, fn in runs.items():
    r = evaluate(fn, qrels, k=3)
    print(f"{name:8} {r['recall@3']:>9.3f} {r['mrr']:>6.3f}")

print("\nBy question kind (recall@3):")
for kind in ("lexical", "semantic", "multihop"):
    subset = [q for q in qrels if q["kind"] == kind]
    row = {name: evaluate(fn, subset, k=3)["recall@3"] for name, fn in runs.items()}
    print(f"  {kind:8}", {k: round(v, 2) for k, v in row.items()})

Read the by-kind table: BM25 should shine on **lexical**, dense on **semantic**,
and **hybrid** should be the most consistent across both — which is exactly why
hybrid is the sane default. (`multihop` stays hard for every single-shot
retriever; that's notebook 06.)

### A note on faithfulness

Retrieval metrics aren't the whole story — a grounded answer must actually be
*supported* by what was retrieved. A cheap proxy: check that each answer
sentence has a high-similarity sentence in the context. Real systems use an
NLI/LLM judge, but the idea is the same.


In [ ]:
def faithfulness(answer_text, contexts, emb, thresh=0.5):
    import re
    ctx_sents = [s for c in contexts for s in re.split(r"(?<=[.!?])\s+", c) if s.strip()]
    ans_sents = [s for s in re.split(r"(?<=[.!?])\s+", answer_text) if s.strip()]
    if not ans_sents or not ctx_sents:
        return 0.0
    cv = emb.encode(ctx_sents)
    supported = 0
    for s in ans_sents:
        sim = cv @ emb.encode(s)
        supported += int(sim.max() >= thresh)
    return supported / len(ans_sents)

ctx = [docs["expense-policy"]]
grounded = "Meals abroad are reimbursed up to SGD 90 per day."
hallucd = "Meals abroad are reimbursed up to SGD 250 per day and alcohol is included."
print("grounded  :", round(faithfulness(grounded, ctx, emb), 2))
print("hallucin. :", round(faithfulness(hallucd, ctx, emb), 2))